<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_RSF_cross_val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q scikit-survival

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.8 MB/s eta 0:00:00


In [8]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

from sksurv.metrics import (
    concordance_index_censored,
    brier_score
)

from sksurv.nonparametric import (
    kaplan_meier_estimator
)

In [10]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/combined_survival_model_ready_wout_protocol/"
)

TRAIN_PATH = (
    BASE_DIR
    / "combined_train_survival_model_ready.csv"
)

VALIDATION_PATH = (
    BASE_DIR
    / "combined_validation_survival_model_ready.csv"
)

TEST_PATH = (
    BASE_DIR
    / "combined_test_survival_model_ready.csv"
)

RAW_CLINICAL_PATH = (
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "AML Dataset/aml_target_2018_pub/"
    "data_clinical_patient.txt"
)

OUTPUT_DIR = (
    BASE_DIR
    / "rsf_fairness_cv_sensitivity"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)

Output directory:
/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/combined_survival_model_ready_wout_protocol/rsf_fairness_cv_sensitivity


In [11]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (620, 68)
Validation: (177, 68)
Test: (89, 68)


In [12]:
development_df = pd.concat(
    [
        train_df,
        validation_df
    ],
    ignore_index=True
)

print(
    "Development CV dataset:",
    development_df.shape
)

print(
    "Independent test dataset:",
    test_df.shape
)

Development CV dataset: (797, 68)
Independent test dataset: (89, 68)


In [13]:
DURATION_COL = "OS_DAYS"
EVENT_COL = "OS_EVENT"

PATIENT_ID_COL = "PATIENT_ID"
SAMPLE_ID_COL = "SAMPLE_ID"

EXCLUDED_COLUMNS = [
    PATIENT_ID_COL,
    SAMPLE_ID_COL,
    DURATION_COL,
    EVENT_COL
]

In [14]:
candidate_features = [
    col
    for col in development_df.columns
    if col not in EXCLUDED_COLUMNS
]

In [15]:
candidate_features = [
    col
    for col in candidate_features
    if pd.api.types.is_numeric_dtype(
        development_df[col]
    )
]

In [16]:
zero_variance_features = [
    feature
    for feature in candidate_features
    if development_df[
        feature
    ].nunique(
        dropna=True
    ) <= 1
]

ORIGINAL_FEATURES = [
    feature
    for feature in candidate_features
    if feature not in zero_variance_features
]

print(
    "Original feature count:",
    len(ORIGINAL_FEATURES)
)

Original feature count: 64


In [17]:
SENSITIVE_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if (
        feature == "AGE"
        or feature.startswith("SEX_")
        or feature.startswith("RACE_")
    )
]

print(
    "Features removed from "
    "sensitivity model:"
)

for feature in SENSITIVE_FEATURES:
    print("-", feature)

Features removed from sensitivity model:
- AGE
- SEX_Female
- SEX_Male
- RACE_American Indian or Alaska Native
- RACE_Asian
- RACE_Black or African American
- RACE_Native Hawaiian or other Pacific Islander
- RACE_Other
- RACE_White


In [18]:
SENSITIVITY_FEATURES = [
    feature
    for feature in ORIGINAL_FEATURES
    if feature not in SENSITIVE_FEATURES
]

print()
print(
    "Original features:",
    len(ORIGINAL_FEATURES)
)

print(
    "Without Age/Sex/Race:",
    len(SENSITIVITY_FEATURES)
)


Original features: 64
Without Age/Sex/Race: 55


In [19]:
assert "AGE" not in SENSITIVITY_FEATURES

assert not any(
    feature.startswith("SEX_")
    for feature in SENSITIVITY_FEATURES
)

assert not any(
    feature.startswith("RACE_")
    for feature in SENSITIVITY_FEATURES
)

print(
    "Age / Sex / Race successfully removed."
)

Age / Sex / Race successfully removed.


In [20]:
raw_df = pd.read_csv(
    RAW_CLINICAL_PATH,
    sep="\t",
    comment="#"
)

demographic_lookup = (
    raw_df[
        [
            "PATIENT_ID",
            "AGE",
            "SEX",
            "RACE"
        ]
    ]
    .drop_duplicates(
        "PATIENT_ID"
    )
    .rename(
        columns={
            "AGE":
                "AGE_ORIGINAL",
            "SEX":
                "SEX_GROUP",
            "RACE":
                "RACE_GROUP"
        }
    )
)

In [21]:
development_df = (
    development_df.merge(
        demographic_lookup,
        on="PATIENT_ID",
        how="left",
        validate="one_to_one"
    )
)

In [22]:
print(
    "Missing age:",
    development_df[
        "AGE_ORIGINAL"
    ].isna().sum()
)

print(
    "Missing sex:",
    development_df[
        "SEX_GROUP"
    ].isna().sum()
)

print(
    "Missing race:",
    development_df[
        "RACE_GROUP"
    ].isna().sum()
)

Missing age: 0
Missing sex: 0
Missing race: 0


In [23]:
development_df[
    "AGE_GROUP"
] = pd.qcut(
    development_df[
        "AGE_ORIGINAL"
    ],
    q=3,
    labels=[
        "Younger",
        "Middle",
        "Older"
    ],
    duplicates="drop"
)

print(
    development_df[
        "AGE_GROUP"
    ]
    .value_counts()
    .sort_index()
)

AGE_GROUP
Younger    271
Middle     288
Older      238
Name: count, dtype: int64


In [24]:
age_ranges = (
    development_df
    .groupby(
        "AGE_GROUP",
        observed=True
    )[
        "AGE_ORIGINAL"
    ]
    .agg(
        [
            "min",
            "max",
            "count"
        ]
    )
)

display(age_ranges)

,min,max,count
AGE_GROUP,,,
Younger,1.0,6.0,271
Middle,7.0,14.0,288
Older,15.0,30.0,238


In [25]:
FINAL_RSF_CONFIG = {
    "n_estimators": 500,
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "bootstrap": True,
    "oob_score": False,
    "n_jobs": -1,
    "random_state": 42
}

In [26]:
def prepare_y(df):

    return Surv.from_arrays(
        event=df[
            EVENT_COL
        ].astype(bool).to_numpy(),

        time=df[
            DURATION_COL
        ].astype(float).to_numpy()
    )

In [27]:
def prepare_X(
    df,
    features
):

    X = df[
        features
    ].copy()

    for feature in features:

        X[feature] = pd.to_numeric(
            X[feature],
            errors="raise"
        ).astype(float)

    return X

In [28]:
def show_stratum_counts(
    df,
    group_column
):

    temp = (
        df.groupby(
            [
                EVENT_COL,
                group_column
            ],
            observed=True
        )
        .size()
        .reset_index(
            name="N"
        )
    )

    print(
        f"\nOutcome + "
        f"{group_column}"
    )

    display(temp)


show_stratum_counts(
    development_df,
    "SEX_GROUP"
)

show_stratum_counts(
    development_df,
    "AGE_GROUP"
)

show_stratum_counts(
    development_df,
    "RACE_GROUP"
)


Outcome + SEX_GROUP


,OS_EVENT,SEX_GROUP,N
0,0.0,Female,239
1,0.0,Male,272
2,1.0,Female,146
3,1.0,Male,140



Outcome + AGE_GROUP


,OS_EVENT,AGE_GROUP,N
0,0.0,Younger,178
1,0.0,Middle,192
2,0.0,Older,141
3,1.0,Younger,93
4,1.0,Middle,96
5,1.0,Older,97



Outcome + RACE_GROUP


,OS_EVENT,RACE_GROUP,N
0,0.0,American Indian or Alaska Native,3
1,0.0,Asian,19
2,0.0,Black or African American,42
3,0.0,Native Hawaiian or other Pacific Islander,3
4,0.0,Other,23
5,0.0,Unknown,27
6,0.0,White,394
7,1.0,American Indian or Alaska Native,1
8,1.0,Asian,20
9,1.0,Black or African American,51


In [29]:
def create_stratification_labels(
    df,
    group_column,
    requested_splits=5
):

    combined_labels = (
        df[
            EVENT_COL
        ]
        .astype(str)
        +
        "_"
        +
        df[
            group_column
        ]
        .astype(str)
    )

    counts = (
        combined_labels
        .value_counts()
    )

    min_combined_count = (
        counts.min()
    )


    # ---------------------------------------------------------
    # Best case:
    # stratify by outcome + subgroup
    # ---------------------------------------------------------

    if (
        min_combined_count
        >= requested_splits
    ):

        print(
            f"{group_column}: "
            f"using {requested_splits}-fold "
            f"stratification by "
            f"OUTCOME + SUBGROUP."
        )

        return (
            combined_labels,
            requested_splits,
            "Outcome + subgroup"
        )


    # ---------------------------------------------------------
    # Try fewer folds
    # ---------------------------------------------------------

    if min_combined_count >= 3:

        n_splits = min(
            requested_splits,
            int(min_combined_count)
        )

        print(
            f"{group_column}: "
            f"using {n_splits}-fold "
            f"stratification by "
            f"OUTCOME + SUBGROUP "
            f"because some strata "
            f"are small."
        )

        return (
            combined_labels,
            n_splits,
            "Outcome + subgroup"
        )


    # ---------------------------------------------------------
    # Fallback:
    # stratify by outcome only
    # ---------------------------------------------------------

    outcome_labels = (
        df[
            EVENT_COL
        ].astype(str)
    )

    outcome_counts = (
        outcome_labels
        .value_counts()
    )


    n_splits = min(
        requested_splits,
        int(
            outcome_counts.min()
        )
    )


    print(
        f"{group_column}: "
        f"Outcome + subgroup "
        f"stratification is NOT feasible."
    )

    print(
        f"Falling back to "
        f"{n_splits}-fold "
        f"stratification by "
        f"OUTCOME only."
    )


    return (
        outcome_labels,
        n_splits,
        "Outcome only"
    )

In [30]:
def calculate_cindex(
    events,
    durations,
    risk
):

    try:

        return (
            concordance_index_censored(
                events.astype(bool),
                durations.astype(float),
                risk
            )[0]
        )

    except Exception:

        return np.nan

In [31]:
HORIZONS = {
    "1Y": 365,
    "2Y": 730,
    "3Y": 1095
}

In [32]:
def km_survival_at_time(
    events,
    durations,
    horizon
):

    times, survival = (
        kaplan_meier_estimator(
            events.astype(bool),
            durations.astype(float)
        )
    )

    valid = (
        times <= horizon
    )

    if not np.any(valid):
        return 1.0

    return survival[
        valid
    ][-1]

In [33]:
def run_fairness_cv(
    df,
    group_column,
    original_features,
    sensitivity_features,
    requested_splits=5,
    random_state=42
):

    strat_labels, n_splits, strat_method = (
        create_stratification_labels(
            df,
            group_column,
            requested_splits
        )
    )


    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )


    overall_results = []
    subgroup_results = []
    brier_results = []
    calibration_results = []

    oof_rows = []


    # ========================================================
    # CV LOOP
    # ========================================================

    for fold, (
        train_idx,
        val_idx
    ) in enumerate(
        cv.split(
            df,
            strat_labels
        ),
        start=1
    ):

        print(
            f"\n{group_column} "
            f"— Fold {fold}/{n_splits}"
        )


        fold_train = (
            df.iloc[
                train_idx
            ]
            .copy()
        )

        fold_val = (
            df.iloc[
                val_idx
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Survival arrays
        # ----------------------------------------------------

        y_train_fold = (
            prepare_y(
                fold_train
            )
        )

        y_val_fold = (
            prepare_y(
                fold_val
            )
        )


        # ----------------------------------------------------
        # ORIGINAL FEATURES
        # ----------------------------------------------------

        X_train_original = (
            prepare_X(
                fold_train,
                original_features
            )
        )

        X_val_original = (
            prepare_X(
                fold_val,
                original_features
            )
        )


        # ----------------------------------------------------
        # NO AGE / SEX / RACE
        # ----------------------------------------------------

        X_train_sensitivity = (
            prepare_X(
                fold_train,
                sensitivity_features
            )
        )

        X_val_sensitivity = (
            prepare_X(
                fold_val,
                sensitivity_features
            )
        )


        # ----------------------------------------------------
        # Train original model
        # ----------------------------------------------------

        rsf_original = (
            RandomSurvivalForest(
                **FINAL_RSF_CONFIG
            )
        )

        rsf_original.fit(
            X_train_original,
            y_train_fold
        )


        # ----------------------------------------------------
        # Train sensitivity model
        # ----------------------------------------------------

        rsf_sensitivity = (
            RandomSurvivalForest(
                **FINAL_RSF_CONFIG
            )
        )

        rsf_sensitivity.fit(
            X_train_sensitivity,
            y_train_fold
        )


        # ----------------------------------------------------
        # Risk predictions
        # ----------------------------------------------------

        risk_original = (
            rsf_original.predict(
                X_val_original
            )
        )

        risk_sensitivity = (
            rsf_sensitivity.predict(
                X_val_sensitivity
            )
        )


        fold_val[
            "RISK_ORIGINAL"
        ] = risk_original

        fold_val[
            "RISK_WITHOUT_ASR"
        ] = risk_sensitivity


        # ----------------------------------------------------
        # Overall C-index
        # ----------------------------------------------------

        c_orig = calculate_cindex(
            fold_val[
                EVENT_COL
            ].values,

            fold_val[
                DURATION_COL
            ].values,

            risk_original
        )


        c_sens = calculate_cindex(
            fold_val[
                EVENT_COL
            ].values,

            fold_val[
                DURATION_COL
            ].values,

            risk_sensitivity
        )


        overall_results.append({
            "Fold": fold,
            "Stratification":
                strat_method,
            "Original_Cindex":
                c_orig,
            "Without_ASR_Cindex":
                c_sens,
            "Change":
                c_sens - c_orig
        })


        # ----------------------------------------------------
        # Survival probabilities
        # ----------------------------------------------------

        surv_orig = (
            rsf_original
            .predict_survival_function(
                X_val_original
            )
        )

        surv_sens = (
            rsf_sensitivity
            .predict_survival_function(
                X_val_sensitivity
            )
        )


        for label, horizon in (
            HORIZONS.items()
        ):

            fold_val[
                f"ORIGINAL_SURVIVAL_{label}"
            ] = [
                float(
                    fn(horizon)
                )
                for fn in surv_orig
            ]


            fold_val[
                f"WITHOUT_ASR_SURVIVAL_{label}"
            ] = [
                float(
                    fn(horizon)
                )
                for fn in surv_sens
            ]


        # ----------------------------------------------------
        # Save OOF predictions
        # ----------------------------------------------------

        oof_part = fold_val[
            [
                PATIENT_ID_COL,
                EVENT_COL,
                DURATION_COL,
                "SEX_GROUP",
                "AGE_GROUP",
                "RACE_GROUP",
                "RISK_ORIGINAL",
                "RISK_WITHOUT_ASR",
                "ORIGINAL_SURVIVAL_1Y",
                "ORIGINAL_SURVIVAL_2Y",
                "ORIGINAL_SURVIVAL_3Y",
                "WITHOUT_ASR_SURVIVAL_1Y",
                "WITHOUT_ASR_SURVIVAL_2Y",
                "WITHOUT_ASR_SURVIVAL_3Y"
            ]
        ].copy()

        oof_part[
            "Fold"
        ] = fold

        oof_rows.append(
            oof_part
        )


        # ====================================================
        # SUBGROUP METRICS
        # ====================================================

        for group, subgroup in (
            fold_val.groupby(
                group_column,
                observed=True
            )
        ):

            n = len(subgroup)

            events = int(
                subgroup[
                    EVENT_COL
                ].sum()
            )


            # -----------------------------------------------
            # Subgroup C-index
            # -----------------------------------------------

            if (
                n >= 5
                and events >= 2
            ):

                subgroup_c_orig = (
                    calculate_cindex(
                        subgroup[
                            EVENT_COL
                        ].values,

                        subgroup[
                            DURATION_COL
                        ].values,

                        subgroup[
                            "RISK_ORIGINAL"
                        ].values
                    )
                )


                subgroup_c_sens = (
                    calculate_cindex(
                        subgroup[
                            EVENT_COL
                        ].values,

                        subgroup[
                            DURATION_COL
                        ].values,

                        subgroup[
                            "RISK_WITHOUT_ASR"
                        ].values
                    )
                )

            else:

                subgroup_c_orig = (
                    np.nan
                )

                subgroup_c_sens = (
                    np.nan
                )


            subgroup_results.append({
                "Fold": fold,
                "Group": group,
                "N": n,
                "Events": events,
                "Original_Cindex":
                    subgroup_c_orig,
                "Without_ASR_Cindex":
                    subgroup_c_sens,
                "Change":
                    (
                        subgroup_c_sens
                        -
                        subgroup_c_orig
                    )
            })


            # ===============================================
            # BRIER SCORE
            # ===============================================

            y_subgroup = (
                prepare_y(
                    subgroup
                )
            )


            orig_probs = subgroup[
                [
                    "ORIGINAL_SURVIVAL_1Y",
                    "ORIGINAL_SURVIVAL_2Y",
                    "ORIGINAL_SURVIVAL_3Y"
                ]
            ].to_numpy()


            sens_probs = subgroup[
                [
                    "WITHOUT_ASR_SURVIVAL_1Y",
                    "WITHOUT_ASR_SURVIVAL_2Y",
                    "WITHOUT_ASR_SURVIVAL_3Y"
                ]
            ].to_numpy()


            if n >= 5:

                try:

                    _, bs_orig = (
                        brier_score(
                            y_train_fold,
                            y_subgroup,
                            orig_probs,
                            np.array(
                                [
                                    365.,
                                    730.,
                                    1095.
                                ]
                            )
                        )
                    )


                    _, bs_sens = (
                        brier_score(
                            y_train_fold,
                            y_subgroup,
                            sens_probs,
                            np.array(
                                [
                                    365.,
                                    730.,
                                    1095.
                                ]
                            )
                        )
                    )

                except Exception:

                    bs_orig = [
                        np.nan,
                        np.nan,
                        np.nan
                    ]

                    bs_sens = [
                        np.nan,
                        np.nan,
                        np.nan
                    ]

            else:

                bs_orig = [
                    np.nan,
                    np.nan,
                    np.nan
                ]

                bs_sens = [
                    np.nan,
                    np.nan,
                    np.nan
                ]


            brier_results.append({
                "Fold": fold,
                "Group": group,
                "N": n,
                "Events": events,

                "Original_BS_1Y":
                    bs_orig[0],

                "Without_ASR_BS_1Y":
                    bs_sens[0],

                "Original_BS_2Y":
                    bs_orig[1],

                "Without_ASR_BS_2Y":
                    bs_sens[1],

                "Original_BS_3Y":
                    bs_orig[2],

                "Without_ASR_BS_3Y":
                    bs_sens[2]
            })


            # ===============================================
            # CALIBRATION
            # ===============================================

            calibration_row = {
                "Fold": fold,
                "Group": group,
                "N": n,
                "Events": events
            }


            if n >= 5:

                for label, horizon in (
                    HORIZONS.items()
                ):

                    observed = (
                        km_survival_at_time(
                            subgroup[
                                EVENT_COL
                            ].values,

                            subgroup[
                                DURATION_COL
                            ].values,

                            horizon
                        )
                    )


                    pred_orig = (
                        subgroup[
                            f"ORIGINAL_SURVIVAL_{label}"
                        ].mean()
                    )


                    pred_sens = (
                        subgroup[
                            f"WITHOUT_ASR_SURVIVAL_{label}"
                        ].mean()
                    )


                    calibration_row[
                        f"Original_Error_{label}"
                    ] = abs(
                        pred_orig
                        -
                        observed
                    )


                    calibration_row[
                        f"Without_ASR_Error_{label}"
                    ] = abs(
                        pred_sens
                        -
                        observed
                    )

            else:

                for label in (
                    HORIZONS.keys()
                ):

                    calibration_row[
                        f"Original_Error_{label}"
                    ] = np.nan

                    calibration_row[
                        f"Without_ASR_Error_{label}"
                    ] = np.nan


            calibration_results.append(
                calibration_row
            )


    # ========================================================
    # RETURN
    # ========================================================

    return {
        "overall":
            pd.DataFrame(
                overall_results
            ),

        "subgroup_cindex":
            pd.DataFrame(
                subgroup_results
            ),

        "brier":
            pd.DataFrame(
                brier_results
            ),

        "calibration":
            pd.DataFrame(
                calibration_results
            ),

        "oof_predictions":
            pd.concat(
                oof_rows,
                ignore_index=True
            ),

        "n_splits":
            n_splits,

        "stratification":
            strat_method
    }

In [34]:
sex_cv = run_fairness_cv(
    development_df,
    group_column="SEX_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

SEX_GROUP: using 5-fold stratification by OUTCOME + SUBGROUP.

SEX_GROUP — Fold 1/5

SEX_GROUP — Fold 2/5

SEX_GROUP — Fold 3/5

SEX_GROUP — Fold 4/5

SEX_GROUP — Fold 5/5


In [35]:
age_cv = run_fairness_cv(
    development_df,
    group_column="AGE_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

AGE_GROUP: using 5-fold stratification by OUTCOME + SUBGROUP.

AGE_GROUP — Fold 1/5

AGE_GROUP — Fold 2/5

AGE_GROUP — Fold 3/5

AGE_GROUP — Fold 4/5

AGE_GROUP — Fold 5/5


In [36]:
race_cv = run_fairness_cv(
    development_df,
    group_column="RACE_GROUP",
    original_features=ORIGINAL_FEATURES,
    sensitivity_features=SENSITIVITY_FEATURES,
    requested_splits=5,
    random_state=42
)

RACE_GROUP: Outcome + subgroup stratification is NOT feasible.
Falling back to 5-fold stratification by OUTCOME only.

RACE_GROUP — Fold 1/5

RACE_GROUP — Fold 2/5

RACE_GROUP — Fold 3/5

RACE_GROUP — Fold 4/5

RACE_GROUP — Fold 5/5


In [37]:
def summarize_overall_cv(
    cv_results
):

    df = cv_results[
        "overall"
    ]

    return pd.DataFrame({
        "Metric": [
            "Original RSF",
            "Without Age/Sex/Race"
        ],

        "Mean C-index": [
            df[
                "Original_Cindex"
            ].mean(),

            df[
                "Without_ASR_Cindex"
            ].mean()
        ],

        "SD": [
            df[
                "Original_Cindex"
            ].std(),

            df[
                "Without_ASR_Cindex"
            ].std()
        ]
    })

In [38]:
print("SEX-STRATIFIED CV")
display(
    summarize_overall_cv(
        sex_cv
    ).round(3)
)

print("AGE-STRATIFIED CV")
display(
    summarize_overall_cv(
        age_cv
    ).round(3)
)

print("RACE-STRATIFIED CV")
display(
    summarize_overall_cv(
        race_cv
    ).round(3)
)

SEX-STRATIFIED CV


,Metric,Mean C-index,SD
0,Original RSF,0.660,0.043
1,Without Age/Sex/Race,0.642,0.047


AGE-STRATIFIED CV


,Metric,Mean C-index,SD
0,Original RSF,0.644,0.025
1,Without Age/Sex/Race,0.632,0.018


RACE-STRATIFIED CV


,Metric,Mean C-index,SD
0,Original RSF,0.658,0.032
1,Without Age/Sex/Race,0.641,0.030


In [39]:
def summarize_subgroup_cindex(
    cv_results
):

    df = (
        cv_results[
            "subgroup_cindex"
        ]
    )

    summary = (
        df.groupby(
            "Group",
            observed=True
        )
        .agg(
            N_Folds=(
                "Fold",
                "nunique"
            ),

            Original_Mean=(
                "Original_Cindex",
                "mean"
            ),

            Original_SD=(
                "Original_Cindex",
                "std"
            ),

            Without_ASR_Mean=(
                "Without_ASR_Cindex",
                "mean"
            ),

            Without_ASR_SD=(
                "Without_ASR_Cindex",
                "std"
            ),

            Mean_Change=(
                "Change",
                "mean"
            )
        )
        .reset_index()
    )

    return summary

In [40]:
sex_cindex_summary = (
    summarize_subgroup_cindex(
        sex_cv
    )
)

age_cindex_summary = (
    summarize_subgroup_cindex(
        age_cv
    )
)

race_cindex_summary = (
    summarize_subgroup_cindex(
        race_cv
    )
)

In [41]:
print("SEX")
display(
    sex_cindex_summary.round(3)
)

print("AGE")
display(
    age_cindex_summary.round(3)
)

print("RACE")
display(
    race_cindex_summary.round(3)
)

SEX


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,Female,5,0.638,0.086,0.620,0.078,-0.017
1,Male,5,0.685,0.039,0.668,0.044,-0.017


AGE


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,Middle,5,0.671,0.063,0.660,0.084,-0.011
1,Older,5,0.615,0.083,0.613,0.078,-0.001
2,Younger,5,0.641,0.067,0.632,0.044,-0.009


RACE


,Group,N_Folds,Original_Mean,Original_SD,Without_ASR_Mean,Without_ASR_SD,Mean_Change
0,American Indian or Alaska Native,2,NaN,NaN,NaN,NaN,NaN
1,Asian,5,0.689,0.210,0.657,0.224,-0.032
2,Black or African American,5,0.558,0.094,0.562,0.075,0.004
3,Native Hawaiian or other Pacific Islander,3,NaN,NaN,NaN,NaN,NaN
4,Other,5,0.655,0.351,0.655,0.351,0.000
5,Unknown,5,0.674,0.158,0.691,0.221,0.018
6,White,5,0.662,0.028,0.657,0.032,-0.005


In [43]:
def summarize_brier_cv(
    cv_results
):

    df = cv_results[
        "brier"
    ]

    columns = [
        "Original_BS_1Y",
        "Without_ASR_BS_1Y",
        "Original_BS_2Y",
        "Without_ASR_BS_2Y",
        "Original_BS_3Y",
        "Without_ASR_BS_3Y"
    ]

    summary = (
        df.groupby(
            "Group",
            observed=True
        )[columns]
        .mean()
        .reset_index()
    )

    return summary

In [44]:
sex_brier_summary = (
    summarize_brier_cv(
        sex_cv
    )
)

age_brier_summary = (
    summarize_brier_cv(
        age_cv
    )
)

race_brier_summary = (
    summarize_brier_cv(
        race_cv
    )
)

In [45]:
print("SEX BRIER")
display(
    sex_brier_summary.round(3)
)

print("AGE BRIER")
display(
    age_brier_summary.round(3)
)

print("RACE BRIER")
display(
    race_brier_summary.round(3)
)

SEX BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,Female,0.130,0.131,0.200,0.201,0.208,0.211
1,Male,0.126,0.128,0.175,0.179,0.188,0.192


AGE BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,Middle,0.106,0.105,0.175,0.175,0.189,0.190
1,Older,0.150,0.153,0.208,0.209,0.212,0.215
2,Younger,0.135,0.136,0.187,0.189,0.203,0.204


RACE BRIER


,Group,Original_BS_1Y,Without_ASR_BS_1Y,Original_BS_2Y,Without_ASR_BS_2Y,Original_BS_3Y,Without_ASR_BS_3Y
0,American Indian or Alaska Native,NaN,NaN,NaN,NaN,NaN,NaN
1,Asian,0.142,0.140,0.246,0.247,0.245,0.245
2,Black or African American,0.234,0.244,0.255,0.264,0.269,0.278
3,Native Hawaiian or other Pacific Islander,NaN,NaN,NaN,NaN,NaN,NaN
4,Other,0.109,0.110,0.189,0.186,0.185,0.182
5,Unknown,0.159,0.158,0.239,0.233,0.226,0.222
6,White,0.114,0.114,0.172,0.174,0.186,0.188


In [46]:
def summarize_calibration_cv(
    cv_results
):

    df = cv_results[
        "calibration"
    ]

    columns = [
        "Original_Error_1Y",
        "Without_ASR_Error_1Y",
        "Original_Error_2Y",
        "Without_ASR_Error_2Y",
        "Original_Error_3Y",
        "Without_ASR_Error_3Y"
    ]

    summary = (
        df.groupby(
            "Group",
            observed=True
        )[columns]
        .mean()
        .reset_index()
    )

    return summary

In [47]:
sex_calibration_summary = (
    summarize_calibration_cv(
        sex_cv
    )
)

age_calibration_summary = (
    summarize_calibration_cv(
        age_cv
    )
)

race_calibration_summary = (
    summarize_calibration_cv(
        race_cv
    )
)

In [48]:
print("SEX CALIBRATION ERROR")
display(
    sex_calibration_summary.round(3)
)

print("AGE CALIBRATION ERROR")
display(
    age_calibration_summary.round(3)
)

print("RACE CALIBRATION ERROR")
display(
    race_calibration_summary.round(3)
)

SEX CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,Female,0.015,0.012,0.034,0.035,0.034,0.034
1,Male,0.012,0.013,0.016,0.018,0.012,0.015


AGE CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,Middle,0.065,0.066,0.059,0.058,0.055,0.054
1,Older,0.037,0.044,0.048,0.063,0.058,0.074
2,Younger,0.027,0.027,0.038,0.040,0.030,0.036


RACE CALIBRATION ERROR


,Group,Original_Error_1Y,Without_ASR_Error_1Y,Original_Error_2Y,Without_ASR_Error_2Y,Original_Error_3Y,Without_ASR_Error_3Y
0,American Indian or Alaska Native,NaN,NaN,NaN,NaN,NaN,NaN
1,Asian,0.074,0.069,0.181,0.170,0.167,0.155
2,Black or African American,0.125,0.160,0.101,0.147,0.140,0.174
3,Native Hawaiian or other Pacific Islander,NaN,NaN,NaN,NaN,NaN,NaN
4,Other,0.114,0.111,0.144,0.137,0.115,0.107
5,Unknown,0.094,0.093,0.160,0.149,0.138,0.134
6,White,0.026,0.025,0.026,0.036,0.025,0.038


In [49]:
def make_slide_cindex_table(
    summary,
    subgroup_name
):

    result = summary[
        [
            "Group",
            "Original_Mean",
            "Without_ASR_Mean",
            "Mean_Change"
        ]
    ].copy()

    result.insert(
        0,
        "Subgroup",
        subgroup_name
    )

    result = result.rename(
        columns={
            "Original_Mean":
                "Original",

            "Without_ASR_Mean":
                "−A/S/R",

            "Mean_Change":
                "Change"
        }
    )

    return result

In [50]:
final_slide_table = pd.concat(
    [
        make_slide_cindex_table(
            sex_cindex_summary,
            "Sex"
        ),

        make_slide_cindex_table(
            age_cindex_summary,
            "Age"
        ),

        make_slide_cindex_table(
            race_cindex_summary,
            "Race"
        )
    ],
    ignore_index=True
)

display(
    final_slide_table.round(3)
)

,Subgroup,Group,Original,−A/S/R,Change
0,Sex,Female,0.638,0.620,-0.017
1,Sex,Male,0.685,0.668,-0.017
2,Age,Middle,0.671,0.660,-0.011
3,Age,Older,0.615,0.613,-0.001
4,Age,Younger,0.641,0.632,-0.009
5,Race,American Indian or Alaska Native,NaN,NaN,NaN
6,Race,Asian,0.689,0.657,-0.032
7,Race,Black or African American,0.558,0.562,0.004
8,Race,Native Hawaiian or other Pacific Islander,NaN,NaN,NaN
9,Race,Other,0.655,0.655,0.000
